# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and field ids
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record set(s) in the dataset.")
for rs in record_sets:
    print(f"\nRecordSet: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract each record set as a pandas DataFrame, keyed by their @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
print("Extracting data for record sets:", record_set_ids)

# Depending on dataset size, you may selectively load a subset.
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for RecordSet {rs_id}")
    if not df.empty:
        print(f"Columns: {df.columns.tolist()}")

# Let's pick the first record set for further exploration
first_record_set_id = record_set_ids[0]
print(f"First Record Set ID: {first_record_set_id}")
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Display numeric columns for the first record set
df = dataframes[first_record_set_id]
numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print("Numeric fields:", numeric_fields)
if not numeric_fields:
    raise ValueError("No numeric fields found for EDA. Check record set or field definitions.")

# For demonstration, pick the first numeric field
numeric_field = numeric_fields[0]
print(f"Using '{numeric_field}' as numeric field for filtering and normalization.")

# Filter rows where the numeric_field > a threshold
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize this field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a non-numeric field
group_fields = df.select_dtypes(include=['object']).columns.tolist()
group_field = None
for col in group_fields:
    # Skip index or obviously unique fields
    if df[col].nunique() < df.shape[0] and df[col].nunique() > 1:
        group_field = col
        break
if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped average '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping. Consider using field definitions for categorical grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If we have at least two numeric fields, make a scatterplot
if len(numeric_fields) > 1:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
    plt.title(f"Scatterplot of {numeric_fields[0]} vs {numeric_fields[1]}")
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the metadata and records using the `mlcroissant` library from the Croissant schema.
- The available record sets and fields were discovered programmatically via their `@id` and inspected for numeric and categorical variables.
- The notebook demonstrated how to filter, normalize, and group data using pandas DataFrames, and visualize distributions for exploratory data analysis.

Continue by exploring the field definitions in depth, addressing missing values, or applying machine learning workflows as appropriate for your domain and dataset.